# 03 · Baseline Evaluation
**Goal:** measure the *untrained* model on a frozen test set, so SFT and DPO have a fair "before" number to beat.

This notebook does five things:
1. **Freezes the test set**, so base, SFT and DPO are always scored on the exact same questions.
2. **Sanity-checks the scorer**: gold answers must score 100%.
3. **Rule-based floor**: a dumb keyword heuristic. Any model should beat it.
4. **Base model (Qwen2.5-0.5B-Instruct, zero-shot)**: the real baseline.
5. **General-behaviour check**: plain chat plus "no tool fits" prompts, to catch regressions after SFT/DPO.

**Why a separate notebook?** Evaluation runs three times (base → SFT → DPO). Keeping it apart from the data pipeline means you re-run only this, with a different `MODEL_PATH` and `RUN_NAME`.

**Why everything is imported from `src/`:** the scorer (`src/evaluate.py`), prompt (`src/prompts.py`) and generator (`src/generation.py`) are shared by every stage. If SFT were scored by slightly different code, or trained on a slightly different prompt, the comparison would be meaningless.

## Setup

In [ ]:
import json
import os
import random
import shutil
import stat
import sys
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from src.data_utils import read_jsonl                                    # noqa: E402
from src.evaluate import (                                               # noqa: E402
    HEADLINE, METRICS, breakdown, compare_runs, extract_json, failure_mix,
    normalize_calls, query_template, score_prediction, summarize, tag_features,
)
from src.generation import run_generation                                # noqa: E402
from src.prompts import PROMPT_VERSION, SYSTEM_PROMPT, build_messages    # noqa: E402

print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Config
- `BENCHMARK`: `"test_v1"` (all ~2.9k questions, ±1.8% margin of error) or `"test_v1_1k"` (fixed 1,000-question subset, ±3%, ~3× faster). **Pick one and use the same one for base, SFT and DPO.**
- `RUN_NAME` / `MODEL_PATH`: for SFT and DPO later, only these two lines change.
- `BATCH_SIZE`: 16 should fit easily in 6 GB for a 0.5B model. If memory runs out, it halves automatically.

In [ ]:
BENCHMARK = "test_v1"            # or "test_v1_1k"
RUN_NAME = "base_qwen2.5-0.5b"
MODEL_PATH = "Qwen/Qwen2.5-0.5B-Instruct"

BATCH_SIZE = 16
MAX_NEW_TOKENS = 512

SPLITS_DIR = Path("data/splits")
EVAL_DIR = Path("data/eval")
RESULTS_DIR = EVAL_DIR / "results" / RUN_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Freeze the test set
`data/splits/test.jsonl` is rewritten every time the split cell in the data notebook runs. If it changed between the base and SFT evaluations, the comparison would be invalid without anyone noticing.

So we make a **frozen copy once**:
- `data/eval/test_v1.jsonl` (all questions) and `test_v1_1k.jsonl` (a fixed random 1,000)
- `data/eval/manifest.json` with each file's **SHA-256 hash**
- files are set to **read-only**

Every later run **verifies the hashes** and refuses to continue if anything changed. Need a new test set? Make `test_v2`, never edit v1.

In [ ]:
MANIFEST_PATH = EVAL_DIR / "manifest.json"
FROZEN_FILES = ["test_v1.jsonl", "test_v1_1k.jsonl", "general_prompts_v1.jsonl"]


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def make_read_only(path):
    os.chmod(path, stat.S_IREAD | stat.S_IRGRP | stat.S_IROTH)


if not MANIFEST_PATH.exists():
    print("No frozen test set yet -> creating v1")
    EVAL_DIR.mkdir(parents=True, exist_ok=True)

    shutil.copyfile(SPLITS_DIR / "test.jsonl", EVAL_DIR / "test_v1.jsonl")

    full = read_jsonl(EVAL_DIR / "test_v1.jsonl")
    subset = random.Random(42).sample(sorted(full, key=lambda r: r["id"]), k=min(1000, len(full)))
    with open(EVAL_DIR / "test_v1_1k.jsonl", "w", encoding="utf-8") as f:
        for r in sorted(subset, key=lambda r: r["id"]):
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    assert (EVAL_DIR / "general_prompts_v1.jsonl").exists(), "general_prompts_v1.jsonl missing"

    manifest = {
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source": str(SPLITS_DIR / "test.jsonl"),
        "subset_seed": 42,
        "files": {name: {"sha256": sha256_file(EVAL_DIR / name),
                         "n": sum(1 for _ in open(EVAL_DIR / name, encoding="utf-8"))}
                  for name in FROZEN_FILES},
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
    for name in FROZEN_FILES + ["manifest.json"]:
        make_read_only(EVAL_DIR / name)

manifest = json.loads(MANIFEST_PATH.read_text())
for name, info in manifest["files"].items():
    actual = sha256_file(EVAL_DIR / name)
    assert actual == info["sha256"], f"{name} changed since it was frozen! Restore it or create a new version."

print(f"Frozen benchmark verified ✅ (created {manifest['created_utc']})")
for name, info in manifest["files"].items():
    print(f"  {name:28} n={info['n']:5}  sha256={info['sha256'][:16]}...")

In [ ]:
test_records = read_jsonl(EVAL_DIR / f"{BENCHMARK}.jsonl")
print(f"Benchmark: {BENCHMARK} ({len(test_records):,} questions)")

## 2. Tag each question with its features
Same features as model selection, plus one new one:

- **`template_twin`**: does a near-identical question (only numbers or quoted strings differ) exist in `sft_train`? Example: test *"Is 27 a power of 3?"* vs train *"Is 27 a power of 5?"*. About 13% of test questions have such a twin. Reporting `seen_template` vs `novel` separately shows whether SFT gains come from real generalisation or from having seen the pattern.

In [ ]:
train_templates = {query_template(r["query"]) for r in read_jsonl(SPLITS_DIR / "sft_train.jsonl")}

features = {r["id"]: tag_features(r, train_templates) for r in test_records}
features_df = pd.DataFrame.from_dict(features, orient="index").rename_axis("id").reset_index()

features_df.drop(columns="id").apply(lambda col: col.value_counts().to_dict())

## 3. Scorer sanity check
Before trusting any number: **the gold answer must score 100% on everything**, including when its calls are shuffled (parallel calls have no fixed order). The earlier baseline cells failed exactly this test. They looked for `tool["function"]["name"]` (OpenAI format), but xLAM tools are just `tool["name"]`, so every answer was marked invalid.

In [ ]:
gold_scores = pd.DataFrame([score_prediction(json.dumps(r["answers"]), r) for r in test_records])
shuffled_scores = pd.DataFrame([
    score_prediction(json.dumps(list(reversed(r["answers"]))), r) for r in test_records
])

for name, df in [("gold", gold_scores), ("gold, calls reversed", shuffled_scores)]:
    assert df["exact_match"].all() and df["schema_valid"].all() and (df["argument_f1"] == 1).all(), name
    assert not df["invented_tool"].any() and not df["invented_argument"].any(), name
print("Gold answers score 100%, order-insensitive ✅")

## Metrics
| Metric | Meaning | Role |
|---|---|---|
| **`exact_match`** | every call fully correct, order ignored | **headline** |
| `exact_match_lenient` | same, but `"5"` == `5` | is the gap just types? |
| `function_match` | right tools (and how many), ignoring values | tool selection |
| `argument_f1` | partial credit per argument, calls paired by name | argument filling |
| `parse_valid` | parseable tool-call JSON | format |
| `schema_valid` | real tools, real args, required args present | format + schema |
| `invented_tool` / `invented_argument` | hallucinated tool or argument | hallucination (great DPO target) |

Every number comes with a **95% confidence interval** (bootstrap over questions). Two runs are compared **question by question** (`compare_runs`): how many questions B fixed and broke, plus a McNemar test to see if the difference is real.

## 4. Rule-based floor
Your keyword heuristic (fixed for the xLAM tool format): pick the tool whose name/description words appear in the query, with empty arguments. It can only score exact match on tools that take no arguments, so it's a **floor**: it shows the task isn't solvable by keyword matching, and every model must clear it.

In [ ]:
import re


def rule_based_predict(record):
    query = record["query"].lower()
    best_score, best_tool = 0, None

    for tool in record["tools"]:
        score = sum(1 for tok in re.findall(r"[a-z]+", tool["name"].lower()) if len(tok) > 2 and tok in query)
        score += 0.1 * sum(
            1 for tok in set(re.findall(r"[a-z]+", tool.get("description", "").lower()))
            if len(tok) > 4 and tok in query
        )
        if score > best_score:
            best_score, best_tool = score, tool

    if best_tool is None:
        return "[]"
    return json.dumps([{"name": best_tool["name"], "arguments": {}}])


rule_df = pd.DataFrame([
    {"id": r["id"], "response": rule_based_predict(r), **score_prediction(rule_based_predict(r), r)}
    for r in test_records
]).merge(features_df, on="id")

summarize(rule_df)

## 5. Base model: Qwen2.5-0.5B-Instruct, zero-shot
**Zero-shot, not few-shot:** in model selection, few-shot examples *confused* Qwen (tool selection dropped from 87% to 61%). SFT will train on the zero-shot prompt anyway, so that's the fair baseline.

**Prompt:** `src/prompts.py` (version below). It's the model-selection prompt plus one line telling the model to return `[]` when no tool fits. That line matters for the general-behaviour check, and SFT will train on this exact prompt.

**Speed:** batched + stops when the JSON closes + sorted by length. Results are written after every batch, so you can stop and resume any time.

In [ ]:
print(f"Prompt version: {PROMPT_VERSION}\n")
print(SYSTEM_PROMPT)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype="auto", device_map="auto")
model.eval()

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

RUN_CONFIG = {
    "model": MODEL_PATH, "benchmark": BENCHMARK,
    "benchmark_sha256": manifest["files"][f"{BENCHMARK}.jsonl"]["sha256"],
    "prompt_version": PROMPT_VERSION, "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy",
}

generations = run_generation(
    model, tokenizer, test_records, build_messages,
    out_path=RESULTS_DIR / "predictions.jsonl",
    run_config=RUN_CONFIG,
    batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
)

peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None
print(f"Peak VRAM: {peak_vram_gb:.2f} GB" if peak_vram_gb else "")

## Results

In [ ]:
records_by_id = {r["id"]: r for r in test_records}

base_df = pd.DataFrame([
    {"id": rid, **{k: g[k] for k in ["response", "gen_tokens", "prompt_tokens", "hit_max_tokens", "latency_per_example"]},
     **score_prediction(g["response"], records_by_id[rid])}
    for rid, g in generations.items()
]).merge(features_df, on="id")

assert len(base_df) == len(test_records), "some questions have no prediction"

base_summary = summarize(base_df)
base_summary

### Base vs rule floor
`fixed` = questions the base model gets right that the rule floor got wrong, and `broke` = the reverse. Same format you'll use for base → SFT → DPO.

In [ ]:
comparison = compare_runs(rule_df, base_df)
comparison

In [ ]:
plot_metrics = ["exact_match", "function_match", "argument_f1", "parse_valid", "schema_valid"]
both = pd.DataFrame({"rule floor": summarize(rule_df, plot_metrics)["value"],
                     "base Qwen": base_summary.loc[plot_metrics, "value"]})
err = [base_summary.loc[plot_metrics, "value"] - base_summary.loc[plot_metrics, "ci_low"],
       base_summary.loc[plot_metrics, "ci_high"] - base_summary.loc[plot_metrics, "value"]]

ax = both.plot(kind="bar", figsize=(10, 4.5), rot=0)
ax.errorbar(x=[i + 0.125 for i in range(len(plot_metrics))], y=both["base Qwen"], yerr=err,
            fmt="none", ecolor="black", capsize=3)
ax.set_ylabel("%")
ax.set_title(f"Baseline on {BENCHMARK} (n={len(base_df)}) — error bars: 95% CI")
plt.tight_layout()
plt.show()

### Where does the base model fail?
Breakdowns by dataset feature, using exact match %. These are the rows to compare after SFT and DPO.

In [ ]:
for feature in ["call_pattern", "calls_bucket", "tools_bucket", "arg_complexity", "has_optional", "template_twin"]:
    print(f"\n=== {feature} ===")
    display(breakdown(base_df, feature))

In [ ]:
print("Failure mix (% of questions):")
display(failure_mix(base_df).to_frame("percent"))

print("\nGeneration behaviour:")
display(pd.Series({
    "avg output tokens": base_df["gen_tokens"].mean(),
    "hit max_new_tokens (%)": 100 * base_df["hit_max_tokens"].mean(),
    "avg prompt tokens": base_df["prompt_tokens"].mean(),
    "latency per example (s)": base_df["latency_per_example"].mean(),
}).round(2))

In [ ]:
# eyeball real failures, so you know the scorer is fair and what SFT needs to fix
for ftype, group in base_df[base_df["failure_type"] != "correct"].groupby("failure_type"):
    row = group.iloc[0]
    rec = records_by_id[row["id"]]
    print(f"##### {ftype}  ({len(group)} questions)")
    print("QUERY :", rec["query"])
    print("GOLD  :", json.dumps(rec["answers"], ensure_ascii=False)[:300])
    print("MODEL :", str(row["response"])[:300], "\n")

## 6. General-behaviour check (regression baseline)
Fine-tuning on 48k tool-calling examples can make a model forget how to just *talk*. This small set measures that **now**, so we can compare after SFT/DPO:

- **10 plain chat prompts** (no tools, no system prompt): the model should answer normally. Metric: **% that wrongly come out as tool-call JSON** (should stay ~0).
- **10 "no tool fits" prompts** (tool prompt, but irrelevant tools, e.g. "write a poem" with only `get_weather`): the right answer is `[]`. Metric: **% correctly returning `[]`**. xLAM has no such examples, so expect this to be low and possibly drop after SFT. That's the gap the DPO stage (or irrelevance data) can fix.

20 prompts is too few for precise numbers. The point is to catch big regressions and to read the outputs side by side.

In [ ]:
general = read_jsonl(EVAL_DIR / "general_prompts_v1.jsonl")
chat_prompts = [g for g in general if g["kind"] == "chat"]
irrelevant_prompts = [g for g in general if g["kind"] == "irrelevant_tools"]

chat_out = run_generation(
    model, tokenizer, chat_prompts, lambda r: [{"role": "user", "content": r["query"]}],
    out_path=RESULTS_DIR / "general_chat.jsonl", run_config={**RUN_CONFIG, "set": "chat"},
    batch_size=BATCH_SIZE, max_new_tokens=200, stop_at_json=False,
)
irrelevant_out = run_generation(
    model, tokenizer, irrelevant_prompts, build_messages,
    out_path=RESULTS_DIR / "general_irrelevant.jsonl", run_config={**RUN_CONFIG, "set": "irrelevant"},
    batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
)


def looks_like_tool_call(text):
    calls = normalize_calls(extract_json(text))
    return bool(calls)


chat_tool_rate = 100 * sum(looks_like_tool_call(chat_out[g["id"]]["response"]) for g in chat_prompts) / len(chat_prompts)
irrelevance_acc = 100 * sum(
    normalize_calls(extract_json(irrelevant_out[g["id"]]["response"])) == [] for g in irrelevant_prompts
) / len(irrelevant_prompts)

print(f"Plain chat answered as tool-call JSON: {chat_tool_rate:.0f}%  (lower is better)")
print(f"'No tool fits' correctly returned []:  {irrelevance_acc:.0f}%  (higher is better)")

In [ ]:
general_view = pd.DataFrame(
    [{"kind": "chat", "query": g["query"], "response": chat_out[g["id"]]["response"][:200]} for g in chat_prompts]
    + [{"kind": "irrelevant", "query": g["query"], "response": irrelevant_out[g["id"]]["response"][:200]} for g in irrelevant_prompts]
)
with pd.option_context("display.max_colwidth", 200):
    display(general_view)

## 7. Save
Everything later stages need to compare against:
- `predictions.jsonl`: raw responses (written during generation)
- `scored.csv`: one row per question with every metric and feature, used by `compare_runs` after SFT/DPO
- `metrics.json`: headline numbers, CIs, breakdowns, run config

In [ ]:
base_df.to_csv(RESULTS_DIR / "scored.csv", index=False)
rule_df.to_csv(EVAL_DIR / "results" / "rule_floor_scored.csv", index=False)

metrics_report = {
    "run_name": RUN_NAME,
    "run_config": RUN_CONFIG,
    "n": len(base_df),
    "summary": base_summary.to_dict(orient="index"),
    "rule_floor_exact_match": round(100 * rule_df["exact_match"].mean(), 1),
    "vs_rule_floor": comparison,
    "breakdowns": {
        f: breakdown(base_df, f).to_dict(orient="index")
        for f in ["call_pattern", "calls_bucket", "tools_bucket", "arg_complexity", "has_optional", "template_twin"]
    },
    "failure_mix": failure_mix(base_df).to_dict(),
    "generation": {
        "avg_gen_tokens": round(base_df["gen_tokens"].mean(), 1),
        "hit_max_tokens_pct": round(100 * base_df["hit_max_tokens"].mean(), 1),
        "latency_per_example_s": round(base_df["latency_per_example"].mean(), 3),
        "peak_vram_gb": round(peak_vram_gb, 2) if peak_vram_gb else None,
    },
    "general_behaviour": {"chat_as_tool_call_pct": chat_tool_rate, "irrelevance_returns_empty_pct": irrelevance_acc},
}
(RESULTS_DIR / "metrics.json").write_text(json.dumps(metrics_report, indent=2, default=str))
print(f"Saved to {RESULTS_DIR}")

## Reusing this for SFT and DPO
After training, copy this notebook (or re-run it) with only two changes:
```python
RUN_NAME = "sft_qwen2.5-0.5b"          # then "dpo_qwen2.5-0.5b"
MODEL_PATH = "path/to/merged_sft_model"
```
Then compare question by question:
```python
base = pd.read_csv("data/eval/results/base_qwen2.5-0.5b/scored.csv")
sft  = pd.read_csv("data/eval/results/sft_qwen2.5-0.5b/scored.csv")
compare_runs(base, sft)   # fixed / broke / McNemar p
```
**Rules that keep the comparison valid:** same `BENCHMARK`, same `PROMPT_VERSION`, greedy decoding, same scorer. The frozen-set hash check and `RUN_CONFIG` enforce most of this automatically.